# Generate Benchmark

In [1]:
import sys
from pathlib import Path

project_root_candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next(path for path in project_root_candidates if (path / "pyproject.toml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import logging
from openai import OpenAI

from src.benchmark import BenchmarkGenerator
from src.llm.llm import OpenAILLM
from src.preprocessing import Reader

import os
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

base_url = os.getenv('BASE_OPEN_AI_URL')
api_key = os.getenv("OPEN_AI_KEY")
client = OpenAI(api_key=api_key, base_url=base_url)

/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("benchmark_generation")

llm = OpenAILLM(client=client, model_name="gpt-5.4")
reader = Reader(logger=logger)
generator = BenchmarkGenerator(reader=reader, llm=llm, logger=logger)

experiment_name = "benchmark_v1"
DATA_PATH = "../../../data/raw_data"
pdf_paths = [os.path.join(DATA_PATH, name) for name in os.listdir(DATA_PATH)]

In [15]:
result = generator.generate(
    pdf_paths=pdf_paths,
    experiment_name=experiment_name,
    questions_per_type=50,
    seed=42,
)

print(result["output_dir"])
result["dataframe"].head()

100%|██████████| 175/175 [00:52<00:00,  3.35it/s]
INFO:benchmark_generation:Resuming benchmark generation in /Users/danilaandreev/Documents/HSE/degree/src/benchmark/results/generation_benchmark_v1_20260402_212121
  0%|          | 0/6 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
INFO:benchmark_generation:Generated factual sample 43/50 from ../../../data/raw_data/4293816468.pdf page 77
INFO:httpx:HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
INFO:benchmark_generation:Generated factual sample 44/50 from ../../../data/raw_data/4293747631.pdf page 193
INFO:httpx:HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
INFO:benchmark_generation:Generated factual sample 45/50 from ../../../data/raw_data/4293836414.pdf page 12
INFO:httpx:HTTP Request: POST https://api.vsellm.ru/v1/responses "HTTP/1.1 200 OK"
INFO:benchmark_generation:Generated factual sample 46/50 from ../../../data/raw_data/42948151

/Users/danilaandreev/Documents/HSE/degree/src/benchmark/results/generation_benchmark_v1_20260402_212121


,sample_id,question_type,question,gold_answer,source_path,page_number,source_excerpt,judge_schema_name,criteria,metadata
0,factual-001-7e121cb5,factual,Чему равен взвешивающий коэффициент для гонад ...,"0,20",../../../data/raw_data/4293816468.pdf,77,СП 2.6.1.2612— 10 Единица удельной активности ...,FactualJudgeResult,"{""atomic_facts"": [""Взвешивающий коэффициент дл...","{""generation_model_name"": ""gpt-5.2"", ""source_c..."
1,factual-003-a02fa764,factual,Какой должна быть минимальная ширина проходов ...,Не менее 500 мм.,../../../data/raw_data/4293836414.pdf,12,вычислительными машинами (ПЭВМ) должна быть ос...,FactualJudgeResult,"{""atomic_facts"": [""Ширина проходов в дизель-ге...","{""generation_model_name"": ""gpt-5.2"", ""source_c..."
2,factual-004-ca1eed18,factual,Какое оборудование используется для проведения...,Лебедка,../../../data/raw_data/4294815196.pdf,35,СП 42-103-2003 водов после соответствующей их ...,FactualJudgeResult,"{""atomic_facts"": [""Для проведения работ по про...","{""generation_model_name"": ""gpt-5.2"", ""source_c..."
3,factual-005-c3eccf7c,factual,Чему равен коэффициент условий работы y_c для ...,"1,05",../../../data/raw_data/4293732009.pdf,44,СП 121.13330.2019 в плитах нижнего слоя покрыт...,FactualJudgeResult,"{""atomic_facts"": [""Для групп участков аэродром...","{""generation_model_name"": ""gpt-5.2"", ""source_c..."
4,factual-007-4bf038a4,factual,Кем принимается решение о выполнении инженерны...,"Лицом, осуществляющим подготовку инвестиционны...",../../../data/raw_data/4293747752.pdf,18,"СП 47.13330.2016 - материалов, необходимых для...",FactualJudgeResult,"{""atomic_facts"": [""Решение о выполнении инжене...","{""generation_model_name"": ""gpt-5.2"", ""source_c..."


In [6]:
result["dataframe"]['question_type'].value_counts()

question_type
factual       50
choice        50
multi_fact    50
procedure     50
comparison    50
negative      50
Name: count, dtype: int64

In [ ]:
# for row in result["dataframe"].iterrows():
#     print(row[0])
#     print(f"Q: {row[1]['question']}")
#     print(f"A: {row[1]['gold_answer']}")
#     print()